# Semana 11
## Comparacion transversal, normalizacion y preparacion para mapas

**Objetivo**: construir vistas comparativas rigurosas y preparar datos espaciales para Tableau.

**Herramientas teoricas de la semana**
- comparacion multigrupo
- normalizacion por tasa
- mapas con contexto
- small multiples


### Agenda sugerida de 4 horas
- 0:00 - 0:30: teoria de comparacion transversal y riesgo de mapas
- 0:30 - 1:20: agregaciones por estado y region
- 1:20 - 2:10: normalizacion y small multiples
- 2:10 - 3:20: preparacion de capa espacial para Tableau
- 3:20 - 4:00: discusion metodologica


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from _shared import (
    make_base_sales,
    introduce_quality_issues,
    profile_dataframe,
    clean_sales_data,
    build_star_schema,
    save_for_tableau,
    ensure_output_dir,
    contrast_ratio,
    make_high_dimensional_dataset,
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
WEEK = "week-11"
OUTPUT_DIR = ensure_output_dir(WEEK)

clean, _ = clean_sales_data(introduce_quality_issues(make_base_sales(n=2400, seed=31), seed=31))
state_summary = (
    clean.groupby(['region', 'state', 'latitude', 'longitude'], as_index=False)
    .agg(total_sales=('sales', 'sum'), orders=('order_id', 'count'), avg_profit=('profit', 'mean'))
)
state_summary['sales_per_order'] = state_summary['total_sales'] / state_summary['orders']
state_summary.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=state_summary.sort_values('sales_per_order', ascending=False), x='state', y='sales_per_order', hue='region', palette='Greys', ax=axes[0])
axes[0].set_title('Ventas por orden por estado')
axes[0].tick_params(axis='x', rotation=45)
for region, subset in state_summary.groupby('region'):
    subset = subset.sort_values('total_sales', ascending=False)
    axes[1].plot(subset['state'], subset['total_sales'], marker='o', label=region)
axes[1].set_title('Comparacion tipo small multiple lineal por region')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
plt.tight_layout()


In [ ]:
geo_for_tableau = state_summary[['state', 'region', 'latitude', 'longitude', 'total_sales', 'sales_per_order', 'avg_profit']].copy()
geo_for_tableau


In [ ]:
save_for_tableau(state_summary, WEEK, 'state_summary')
save_for_tableau(geo_for_tableau, WEEK, 'geo_for_tableau')


### Uso teorico de herramientas
- `sales_per_order` traduce la idea de normalización.
- La tabla `geo_for_tableau` deja lista una capa para mapas en Tableau.
- Comparar un mapa con una barra ordenada permite discutir precisión visual.
